# 02 · Preprocessing & Cleaning

**Objective:** apply every fix from the Step 1 issue log (`reports/data_quality_log.csv`) and write analysis-ready tables to `processed/`.

**Principles**
- Raw files in `wellbeingdata/` are never modified.
- Each fix references its issue number (**#n**) from the log.
- Order matters. Timestamps are fixed **before** date filtering and de-duplication, because both depend on correct times.
- Nothing is silently deleted. Every step prints how many rows it removed, and questionable records are **flagged** rather than dropped whenever an analysis might still need them.
- The final section re-runs the Step 1 checks to prove each issue is resolved.

**Outputs (`processed/`)**

| File | Grain |
|---|---|
| `users.parquet` | one row per real user |
| `events.parquet` | one row per de-duplicated event, UTC timestamps, parsed properties |
| `assessments.parquet` | one row per user × wave, correctly scored |
| `experiment.parquet` | one row per eligible experiment user, with reconstructed variant |

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)

RAW = Path("wellbeingdata")
OUT = Path("processed"); OUT.mkdir(exist_ok=True)
CUTOFF = pd.Timestamp("2026-11-15 23:59:59", tz="UTC")

users_raw  = pd.read_csv(RAW / "users.csv", dtype=str)
events_raw = pd.read_csv(RAW / "events.csv", dtype=str)
assess_raw = pd.read_csv(RAW / "assessments.csv", dtype=str)
assign_raw = pd.read_csv(RAW / "experiment_assignments.csv", dtype=str)
content    = pd.read_csv(RAW / "content.csv")

def report(step, before, after):
    print(f"{step:<55s} {before:>9,} -> {after:>9,}  (removed {before - after:,})")

---
## 1. Users
- **#5:** remove internal test accounts. Their channel is `unknown`, which isn't a documented acquisition channel.
- **#1:** de-duplicate the migration copies and keep the legacy row.
- **#2:** standardize platform labels.
- **#3, #4:** make missingness explicit instead of imputing it. The `completed_onboarding` flag is added after the events are cleaned.

In [ ]:
users_raw

,user_id,signup_ts,platform,signup_app_version,country,timezone,age_band,gender,acquisition_channel,meditation_experience,primary_goal,source_system
0,L101349,2026-06-14T10:49:57Z,Web,4.9.0,BR,America/Sao_Paulo,55-64,male,paid_social,none,sleep,legacy
1,u_ae1e9413b8,2026-10-19T15:19:04Z,ios,5.2.0,US,America/Chicago,45-54,female,app_store_search,none,relationships,core
2,u_63976134b9,2026-09-21T07:11:26Z,ios,5.1.0,AU,Australia/Sydney,35-44,female,app_store_search,NaN,NaN,core
3,u_ff2df6e8cf,2026-10-10T20:10:26Z,android,5.2.0,ZA,Africa/Johannesburg,65+,male,partner_org,regular,stress,core
4,u_f48876226a,2026-09-20T16:26:54Z,android,5.0.0,BR,America/Sao_Paulo,NaN,male,paid_social,NaN,NaN,core
...,...,...,...,...,...,...,...,...,...,...,...,...
15141,u_87a79d05f3,2026-08-31T20:53:17Z,ios,5.0.0,ZA,Africa/Johannesburg,25-34,male,app_store_search,regular,general_wellbeing,core
15142,u_a5ed931f8c,2026-10-12T17:15:12Z,android,5.1.1,US,America/Denver,25-34,female,paid_social,some,sleep,core
15143,u_816c5ce306,2026-08-15T08:34:01Z,ios,5.0.0,US,America/New_York,35-44,female,organic,none,relationships,core
15144,L109412,2026-06-03T19:44:33Z,iPhone,4.8.0,US,America/Los_Angeles,65+,male,organic,some,stress,legacy


In [ ]:
# check unique user_id in users_raw
users_raw['user_id'].nunique()

15006

In [ ]:
# print duplicate user_id
users_raw[users_raw.duplicated('user_id', keep=False)]

,user_id,signup_ts,platform,signup_app_version,country,timezone,age_band,gender,acquisition_channel,meditation_experience,primary_goal,source_system
43,L107714,2026-06-09T13:01:17Z,Android,4.8.0,US,America/Chicago,18-24,nonbinary,app_store_search,none,general_wellbeing,legacy
97,L102319,2026-06-21T20:43:13Z,ios,4.8.0,IN,Asia/Kolkata,18-24,male,organic,none,focus,core
164,L109877,2026-06-15T21:43:50Z,web,4.9.0,MX,America/Mexico_City,55-64,female,app_store_search,regular,focus,core
206,L102891,2026-06-27T07:06:23Z,web,4.9.0,FR,Europe/Paris,65+,female,referral,NaN,NaN,core
216,L114887,2026-06-01T14:19:10Z,Android,4.8.0,US,America/New_York,18-24,female,referral,none,stress,legacy
...,...,...,...,...,...,...,...,...,...,...,...,...
14916,L109844,2026-06-12T17:57:26Z,iPhone,4.8.0,US,America/Denver,35-44,female,app_store_search,some,general_wellbeing,legacy
14927,L102273,2026-06-01T07:24:06Z,ios,4.8.0,BR,America/Sao_Paulo,18-24,female,organic,some,stress,core
14971,L109071,2026-06-14T12:04:00Z,android,4.8.0,US,America/Los_Angeles,NaN,female,paid_social,NaN,NaN,core
15074,L109877,2026-06-15T21:42:39Z,Web,4.9.0,MX,America/Mexico_City,55-64,female,app_store_search,regular,focus,legacy


In [ ]:
PLATFORM_MAP = {"iPhone": "ios", "ios": "ios", "Android": "android", "android": "android", "Web": "web", "web": "web"}

u = users_raw.copy()
INTERNAL_IDS = set(u.loc[u.acquisition_channel == "unknown", "user_id"])          # #5
n = len(u); u = u[~u.user_id.isin(INTERNAL_IDS)]; report("#5 remove internal test accounts", n, len(u))

n = len(u)                                                                         # #1
u = (u.assign(_legacy_first=(u.source_system != "legacy").astype(int))
       .sort_values(["user_id", "_legacy_first"])
       .drop_duplicates("user_id", keep="first")
       .drop(columns="_legacy_first"))
report("#1 drop migration duplicates (keep legacy row)", n, len(u))

u["platform"] = u.platform.map(PLATFORM_MAP)                                       # #2
u["signup_ts"] = pd.to_datetime(u.signup_ts, utc=True)
u["age_band"] = u.age_band.fillna("unknown")                                       # #4
for c in ["meditation_experience", "primary_goal"]:                                # #3
    u[c] = u[c].fillna("not_answered")

assert u.user_id.is_unique and u.platform.notna().all()
u.platform.value_counts()

#5 remove internal test accounts                           15,146 ->    15,140  (removed 6)
#1 drop migration duplicates (keep legacy row)             15,140 ->    15,000  (removed 140)


platform
ios        7435
android    5763
web        1802
Name: count, dtype: int64

---
## 2. Events
### 2a. Drop unusable rows and standardize labels
- **#10:** events with no `user_id` (pre-login `app_open`) can't be attributed to anyone.
- **#5:** remove events from internal test accounts.
- **#9:** standardize platform labels.
- **#7:** rename tracking-plan v1 events to their v2 names.

In [ ]:
e = events_raw.copy()
n = len(e); e = e[e.user_id.notna()];                 report("#10 drop events without user_id", n, len(e))
n = len(e); e = e[~e.user_id.isin(INTERNAL_IDS)];     report("#5 drop internal-account events", n, len(e))
n = len(e); e = e[e.user_id.isin(set(u.user_id))];    report("drop events from unknown users", n, len(e))

e["platform"] = e.platform.map(PLATFORM_MAP)                                        # #9
V1_TO_V2 = {"meditation_started": "practice_start", "meditation_completed": "practice_complete"}
e["event_name"] = e.event_name.replace(V1_TO_V2)                                    # #7
e.event_name.value_counts()

#10 drop events without user_id                           603,906 ->   603,242  (removed 664)
#5 drop internal-account events                           603,242 ->   563,780  (removed 39,462)
drop events from unknown users                            563,780 ->   563,780  (removed 0)


event_name
practice_start          170736
app_open                166539
practice_complete       139769
notification_open        18855
assessment_complete      15482
app_install              15266
onboarding_start         15253
onboarding_complete      13119
reminder_set              4501
goal_selected             2023
plan_created              2018
notification_opt_out       219
Name: count, dtype: int64

### 2b. Parse properties JSON into columns (#7)
v1 uses `content_id` / `duration_sec`. v2 uses `contentId` / `durationMs`, which is in **milliseconds**. Both are unified into `content_id` and `duration_sec`.

In [ ]:
props = e.properties.map(json.loads)
e["content_id"]    = props.map(lambda p: p.get("content_id", p.get("contentId")))
e["duration_sec"]  = props.map(lambda p: p["duration_sec"] if "duration_sec" in p
                                         else (p["durationMs"] / 1000 if "durationMs" in p else np.nan))
e["flow"]          = props.map(lambda p: p.get("flow"))
e["wave"]          = props.map(lambda p: p.get("wave"))
e["reminder_time"] = props.map(lambda p: p.get("reminder_time"))

# sanity check: durations should now be on the same scale in both schema versions
e[e.event_name == "practice_complete"].groupby("schema_version").duration_sec.median()

schema_version
1    605.0
2    601.0
Name: duration_sec, dtype: float64

### 2c. Fix timestamps (#6), *before* any date logic
Legacy events (format `YYYY-MM-DD HH:MM:SS`, no `Z`) were stored in the **user's local time**. They're localized with `users.timezone`, then converted to UTC. All other events are already ISO-8601 UTC.

In [ ]:
tz_of = u.set_index("user_id").timezone
naive = ~e.event_ts.str.endswith("Z")
print(f"naive (legacy local-time) events: {naive.sum():,}")

# UTC rows parse directly
parts = [pd.to_datetime(e.loc[~naive, "event_ts"], format="ISO8601", utc=True)]

# naive rows: localize per user timezone, then convert to UTC
tzs = e.loc[naive, "user_id"].map(tz_of)
for tz, idx in tzs.groupby(tzs).groups.items():
    local = pd.to_datetime(e.loc[idx, "event_ts"], format="%Y-%m-%d %H:%M:%S")
    parts.append(local.dt.tz_localize(tz, ambiguous="NaT", nonexistent="shift_forward").dt.tz_convert("UTC"))

e["ts"] = pd.concat([p.dt.as_unit("ms") for p in parts])   # aligns on the original row index
print("unparsed:", e.ts.isna().sum())

naive (legacy local-time) events: 31,891
unparsed: 0


In [ ]:
# Proof the fix is right: for legacy users, app_install should coincide with signup_ts (stored in UTC)
chk = (e[(e.event_name == "app_install") & naive]
       .merge(u[["user_id", "signup_ts"]], on="user_id"))
offset_h = (chk.ts - chk.signup_ts).dt.total_seconds() / 3600
print("legacy app_install minus signup (hours), after fix:"); print(offset_h.round(2).value_counts().head())
# the few huge offsets are clock-skewed rows (years 2001/2027), removed in the next step

legacy app_install minus signup (hours), after fix:
 0.0         1948
 8760.0         4
-219144.0       4
Name: count, dtype: int64


### 2d. Remove impossible dates (#11)
Valid events fall between the user's signup (minus a 5-minute tolerance) and the data cutoff. Rows dated 2001 or 2027 come from clock skew on the device.

In [ ]:
e = e.merge(u[["user_id", "signup_ts"]], on="user_id", how="left")
n = len(e)
e = e[(e.ts >= e.signup_ts - pd.Timedelta(minutes=5)) & (e.ts <= CUTOFF)]
report("#11 drop events outside [signup, cutoff]", n, len(e))

#11 drop events outside [signup, cutoff]                  563,780 ->   562,020  (removed 1,760)


C:\Users\ramba\AppData\Local\Temp\ipykernel_31220\2227139788.py:3: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  e = e[(e.ts >= e.signup_ts - pd.Timedelta(minutes=5)) & (e.ts <= CUTOFF)]


### 2e. Remove client-retry duplicates (#8)
A retry is an identical event (same user, event, session and properties) sent again within 3 seconds. Only the first copy is kept. Exact duplicates are the special case of a 0-second gap.

In [ ]:
KEY = ["user_id", "event_name", "session_id", "properties"]
e = e.sort_values(KEY + ["ts"])
gap = e.groupby(KEY, dropna=False).ts.diff().dt.total_seconds()
n = len(e); e = e[~gap.between(0, 3)]
report("#8 drop retry duplicates (<=3s)", n, len(e))
DUPS_REMAINING = e.sort_values(KEY + ["ts"]).groupby(KEY, dropna=False).ts.diff().dt.total_seconds().between(0, 3).sum()
print("retry duplicates remaining:", DUPS_REMAINING)

#8 drop retry duplicates (<=3s)                           562,020 ->   551,903  (removed 10,117)
retry duplicates remaining: 0


### 2f. Flag the iOS tracking gap (#12)
From Oct 5–12, iOS app 5.2.0 didn't send `practice_complete`. The affected events are **flagged, not dropped**, so features in Step 3 can fall back to `practice_start` for those user-days.

In [ ]:
GAP_START, GAP_END = pd.Timestamp("2026-10-05", tz="UTC"), pd.Timestamp("2026-10-13", tz="UTC")
e["in_ios_tracking_gap"] = (e.platform == "ios") & (e.app_version == "5.2.0") & e.ts.between(GAP_START, GAP_END, inclusive="left")
print("events in gap:", e.in_ios_tracking_gap.sum())
e[e.in_ios_tracking_gap].event_name.value_counts().head()

events in gap: 14987


event_name
practice_start         6200
app_open               5980
notification_open       736
assessment_complete     512
app_install             495
Name: count, dtype: int64

In [ ]:
events = (e.drop(columns=["event_ts", "properties", "signup_ts"])
           .sort_values(["user_id", "ts"]).reset_index(drop=True))
events = events[["event_id", "user_id", "event_name", "ts", "session_id", "platform", "app_version",
                 "schema_version", "content_id", "duration_sec", "flow", "wave", "reminder_time", "in_ios_tracking_gap"]]

# finish users: completed_onboarding flag (#3)
# onboarding is complete if the event exists OR the onboarding answers exist
# (a few users' onboarding_complete events were dropped by the clock-skew filter in 2d)
u["completed_onboarding"] = (u.user_id.isin(set(events.loc[events.event_name == "onboarding_complete", "user_id"]))
                             | (u.meditation_experience != "not_answered"))
print(pd.crosstab(u.completed_onboarding, u.meditation_experience == "not_answered"))
print(events.shape)

meditation_experience  False  True 
completed_onboarding               
False                      0   2099
True                   12901      0
(551903, 14)


---
## 3. Assessments
- **#13:** map `intake` to `baseline`.
- **#14:** treat 99 (skipped) and out-of-range values as missing.
- **#17:** reverse q04/q08/q12/q16 **for v1 only**, because v2 already stores them reversed.
- **#15:** keep the latest submission per user and wave.
- **#16:** flag low-effort responses, meaning straight-lined **and** completed in under 40 seconds.
- Scoring: a pillar is the mean of its items, computed only if ≥ 3 of 4 items are present. The Flourishing Index is the mean of the 4 pillars. PHQ-4 is the sum of its items (0–12).

In [ ]:
ITEMS = [f"q{i:02d}" for i in range(1, 17)]
PHQ = [f"phq4_{i}" for i in range(1, 5)]
REVERSE = ["q04", "q08", "q12", "q16"]
PILLARS = {"awareness": ITEMS[0:4], "connection": ITEMS[4:8], "insight": ITEMS[8:12], "purpose": ITEMS[12:16]}

a = assess_raw.copy()
a = a[a.user_id.isin(set(u.user_id))]
a["wave"] = a.wave.replace({"intake": "baseline"})                                   # #13
a["submitted_ts"] = pd.to_datetime(a.submitted_ts, utc=True)
a["completion_seconds"] = a.completion_seconds.astype(int)
a[ITEMS + PHQ] = a[ITEMS + PHQ].astype(int)

a[ITEMS] = a[ITEMS].where((a[ITEMS] >= 1) & (a[ITEMS] <= 5))                         # #14
a[PHQ] = a[PHQ].where((a[PHQ] >= 0) & (a[PHQ] <= 3))
print("missing item cells:", a[ITEMS].isna().sum().sum())

v1 = a.scoring_version == "v1"                                                       # #17
a.loc[v1, REVERSE] = 6 - a.loc[v1, REVERSE]

n = len(a)                                                                           # #15
a = a.sort_values("submitted_ts").drop_duplicates(["user_id", "wave"], keep="last")
report("#15 keep latest submission per user x wave", n, len(a))

a["low_effort"] = (a[ITEMS].std(axis=1) == 0) & (a.completion_seconds < 40)          # #16
print("low-effort flagged:", a.low_effort.sum())

missing item cells: 3848
#15 keep latest submission per user x wave                 15,574 ->    15,194  (removed 380)
low-effort flagged: 396


In [ ]:
for p, cols in PILLARS.items():
    a[p] = a[cols].mean(axis=1).where(a[cols].notna().sum(axis=1) >= 3)
a["flourishing"] = a[list(PILLARS)].mean(axis=1).where(a[list(PILLARS)].notna().all(axis=1))
a["phq4_total"] = a[PHQ].sum(axis=1).where(a[PHQ].notna().all(axis=1))
a["phq4_anxiety"] = a[["phq4_1", "phq4_2"]].sum(axis=1, min_count=2)
a["phq4_depression"] = a[["phq4_3", "phq4_4"]].sum(axis=1, min_count=2)

a = a.merge(u[["user_id", "signup_ts"]], on="user_id")
a["days_since_signup"] = (a.submitted_ts - a.signup_ts).dt.total_seconds() / 86400
assessments = a.drop(columns="signup_ts").reset_index(drop=True)
assessments.groupby("wave")[["flourishing", "phq4_total"]].describe().round(2)

flourishing                                         phq4_total                                      
               count  mean   std  min   25%   50%   75%  max      count  mean   std  min  25%  50%  75%   max
wave                                                                                                         
baseline     10991.0  3.00  0.71  1.0  2.50  3.00  3.50  5.0    10469.0  4.26  2.84  0.0  2.0  4.0  6.0  12.0
week4         2741.0  3.30  0.75  1.0  2.81  3.31  3.88  5.0     2619.0  3.66  2.80  0.0  1.0  3.0  6.0  12.0
week8         1374.0  3.37  0.76  1.0  2.88  3.44  3.94  5.0     1301.0  3.47  2.78  0.0  1.0  3.0  5.0  12.0

---
## 4. Experiment
- **#18:** drop exact duplicate assignment rows.
- **#19:** keep each user's **first** assignment, which is intention-to-treat.
- **#20:** rebuild the variant for **every eligible user** from `onboarding_start.flow`, which is logged whether or not the assignment was. `assignment_logged` marks the 189 users the assignment log missed.

In [ ]:
asg = assign_raw.drop_duplicates(subset=["user_id", "variant", "assigned_ts"])         # #18
asg["assigned_ts"] = pd.to_datetime(asg.assigned_ts, utc=True)
n = len(asg); asg = asg.sort_values("assigned_ts").drop_duplicates("user_id", keep="first")  # #19
report("#19 keep first assignment per user", n, len(asg))

eligible = u[(u.signup_ts >= "2026-09-01") & (u.signup_ts < "2026-10-01") & u.platform.isin(["ios", "android"])].copy()
flow = (events[events.event_name == "onboarding_start"].sort_values("ts")
        .drop_duplicates("user_id").set_index("user_id").flow)
eligible["variant_from_flow"] = eligible.user_id.map(flow).map({"personal_plan": "treatment", "standard": "control"})
eligible = eligible.merge(asg[["user_id", "variant"]], on="user_id", how="left")

logged = eligible.variant.notna()
print("logged assignments that agree with onboarding flow:",
      f"{(eligible.loc[logged, 'variant'] == eligible.loc[logged, 'variant_from_flow']).mean():.1%}")

eligible["assignment_logged"] = logged
eligible["variant"] = eligible.variant.fillna(eligible.variant_from_flow)               # #20
experiment = eligible[["user_id", "variant", "assignment_logged", "signup_ts", "platform", "signup_app_version",
                       "country", "acquisition_channel", "meditation_experience", "completed_onboarding"]]
pd.crosstab([experiment.platform, experiment.signup_app_version], [experiment.variant, experiment.assignment_logged])

#19 keep first assignment per user                          4,255 ->     4,227  (removed 28)
logged assignments that agree with onboarding flow: 99.7%


C:\Users\ramba\AppData\Local\Temp\ipykernel_31220\1844888784.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  asg["assigned_ts"] = pd.to_datetime(asg.assigned_ts, utc=True)


variant                     control treatment      
assignment_logged             True      False True 
platform signup_app_version                        
android  4.9.0                   23         0    19
         5.0.0                  263         0   245
         5.1.0                  437       189   231
         5.1.1                  290         0   254
ios      4.9.0                   35         0    35
         5.0.0                  345         0   355
         5.1.0                  848         0   847

---
## 5. Validation: re-run the Step 1 checks on the clean data
Every check should pass. If one fails, the corresponding fix needs revisiting.

In [ ]:
checks = {}
checks["#1 users unique"] = u.user_id.is_unique
checks["#5 no internal accounts"] = not (set(u.user_id) | set(events.user_id)) & INTERNAL_IDS
checks["#2/#9 platform labels clean"] = set(u.platform) | set(events.platform) <= {"ios", "android", "web"}
checks["#7 no v1 event names"] = not events.event_name.isin(V1_TO_V2).any()
med = events[events.event_name == "practice_complete"].groupby("schema_version").duration_sec.median()
checks["#7 duration units consistent"] = med.max() / med.min() < 1.5
checks["#6 legacy install == signup (median |h| < 0.1)"] = offset_h.abs().median() < 0.1
checks["#11 dates within window"] = events.ts.between(pd.Timestamp("2026-05-31", tz="UTC"), CUTOFF).all()
checks["#8 no retry duplicates"] = DUPS_REMAINING == 0
checks["#13/#15 one row per user x wave"] = not assessments.duplicated(["user_id", "wave"]).any()
checks["#14 items in 1-5"] = assessments[ITEMS].stack().dropna().between(1, 5).all()

def item_rest(df):
    out = {}
    for cols in PILLARS.values():
        for q in cols:
            out[q] = df[q].corr(df[[c for c in cols if c != q]].mean(axis=1))
    return pd.Series(out)
ir = item_rest(assessments[~assessments.low_effort])
checks["#17 all item-rest correlations > 0.3"] = (ir > 0.3).all()
counts = experiment.variant.value_counts()
checks["#20 SRM passes (p > 0.01)"] = stats.chisquare(counts.values).pvalue > 0.01

pd.Series(checks, name="passed").to_frame()

,passed
#1 users unique,True
#5 no internal accounts,True
#2/#9 platform labels clean,True
#7 no v1 event names,True
#7 duration units consistent,True
#6 legacy install == signup (median |h| < 0.1),True
#11 dates within window,True
#8 no retry duplicates,True
#13/#15 one row per user x wave,True
#14 items in 1-5,True


In [ ]:
print("Item-rest correlations (clean):"); print(ir.round(2).to_string())
print("\nExperiment split:", counts.to_dict(), f"| SRM p = {stats.chisquare(counts.values).pvalue:.2f}")

Item-rest correlations (clean):
q01    0.67
q02    0.67
q03    0.66
q04    0.66
q05    0.65
q06    0.65
q07    0.66
q08    0.65
q09    0.67
q10    0.66
q11    0.67
q12    0.65
q13    0.66
q14    0.66
q15    0.67
q16    0.65

Experiment split: {'control': 2241, 'treatment': 2175} | SRM p = 0.32


## 6. Save

In [ ]:
u.to_parquet(OUT / "users.parquet", index=False)
events.to_parquet(OUT / "events.parquet", index=False)
assessments.to_parquet(OUT / "assessments.parquet", index=False)
experiment.to_parquet(OUT / "experiment.parquet", index=False)
content.to_parquet(OUT / "content.parquet", index=False)
for f in sorted(OUT.glob("*.parquet")):
    print(f"{f.name:<22s} {pd.read_parquet(f).shape}")

assessments.parquet    (15194, 36)
content.parquet        (120, 5)
events.parquet         (551903, 14)
experiment.parquet     (4416, 10)
users.parquet          (15000, 13)


In [ ]:
# pd.read_parquet("processed/events.parquet", columns=["user_id", "event_name", "ts"])